<img src=../figures/Brown_logo.svg width=50%>

## Data-Driven Design & Analyses of Structures & Materials (3dasm)

## Lecture 19.1

### Miguel A. Bessa | <a href = "mailto: miguel_bessa@brown.edu">miguel_bessa@brown.edu</a>  | Associate Professor

### Elvis Aguero | <a href = "mailto: elvis_vera@brown.edu">elvis_vera@brown.edu</a>  | PhD candidate


**What:** A lecture of the "3dasm" course

**Where:** This notebook comes from this [repository](https://github.com/bessagroup/3dasm_course)

**Reference for entire course:** Murphy, Kevin P. *Probabilistic machine learning: an
introduction*. MIT press, 2022. Available online [here](https://probml.github.io/pml-book/book1.html)

**How:** We try to follow Murphy's book closely, but the sequence of Chapters and Sections is
different. The intention is to use notebooks as an introduction to the topic and Murphy's book
as a resource.
* If working offline: Go through this notebook and read the book.
* If attending class in person: listen to me (!) but also go through the notebook in your laptop at the same time. Read the book.
* If attending lectures remotely: listen to me (!) via Zoom and (ideally) use two screens where you have the notebook open in 1 screen and you see the lectures on the other. Read the book.

This is the first of two lectures on **adda**. Today: how you build such a framework, and what
we learned building it. Lecture 19.2: what happened when we pointed it at a real problem.

## **OPTION 1**. Run this notebook **locally in your computer**:
1. Confirm that you have the '3dasm' mamba (or conda) environment (see Lecture 1).
2. Go to the 3dasm_course folder in your computer and pull the last updates of the [repository](https://github.com/bessagroup/3dasm_course):
```
git pull
```
    - Note: if you can't pull the repo due to conflicts (and you can't handle these conflicts), use this command (with **caution**!) and your repo becomes the same as the one online:
```
git reset --hard origin/main
```
3. Open command window and load jupyter notebook (it will open in your internet browser):
```
jupyter notebook
```
5. Open notebook of this Lecture and choose the '3dasm' kernel.

## **OPTION 2**. Use **Google's Colab** (no installation required, but times out if idle):

1. go to https://colab.research.google.com
2. login
3. File > Open notebook
4. click on Github (no need to login or authorize anything)
5. paste the git link: https://github.com/bessagroup/3dasm_course
6. click search and then click on the notebook for this Lecture.

In [1]:
# Basic plotting tools needed in Python.

import matplotlib.pyplot as plt # import plotting tools to create figures
import numpy as np # import numpy to handle a lot of things!

%config InlineBackend.figure_format = "retina" # render higher resolution images in the notebook
plt.rcParams["figure.figsize"] = (8,4) # rescale figure size appropriately for slides

# To limit the number of rows to show in a dataframe, for presentation purposes:
import pandas as pd

pd.set_option('display.max_rows', 10)

In [2]:
# In Google Colab you need to install f3dasm first (locally it is already in the '3dasm'
# environment). Uncomment the line below if you are running in Colab:

# %pip install f3dasm

from f3dasm import ExperimentData   # the same object you used in Lectures 17, 18 and 19

## Outline for today

* What an agent is: tools, nodes, a graph
* Why more than one node: safety, specialization, efficiency
* Where the state lives
* The decisions you have to make, and what defends them
* What admits a claim
* Getting started, and an in-class exercise

**Reading material**: this notebook + the
[a3dasm documentation](https://elvis-aguero.github.io/a3dasm/).


## Agents and tools

* A language model emits text. They can also run a script, read output, edit a file.

* A model with  a set of tools, and a job is an **agent** — a **subagent** when another agent spawns it. As of 2026, they are already automatically dispatched in current AI coding agents (Codex, Claude Code, Opencode, Kimi Code, Antigravity). 

* A fixed code path that calls a model at each step is a **workflow**. We can **orchestrate** multiple of them.

## Agent's as a graph

There are multiple ways to arrange multiple agents to collaborate. Some people are trying:

* Group chats of agents
* (Other ways of )
* A graph architecture where each node is a possibly different agent, and edges dictate who can talk to who.

We will show you the decisions involved when architecting a graph of agents. 

## Why more than one node

Three principles: Safety, specialization, efficiency. 

We will see that there are a few advantages of arranging agents in a graph format.

## Specialization

Different jobs, different models. A strong model plans, a cheaper one executes.

A fine-tuned open-weights model already is on par with frontier models on narrow tasks at a fraction of the compute. Each node can get its own backend with their own settings.

## Separation of privilege

Each node gets a specialized set of tools tailored to their goals.

## Failure isolation and parallelism

Work is handed over in units you can cancel, retry, or abandon. A six-hour run that dies at hour five does not start again.

Independent units also run at the same time.

## When one chat is enough

Short task. Cheap to check. You already know the plan.

Then a single chat is the right tool, and a graph is overhead.

## Where the state lives

A node is stateless between calls.

Agent frameworks typically checkpoint the whole conversation, so a thread can be resumed.
We chose to make the record the state instead: that buys reproducibility rather than
resumable dialogue. (what does it mean for the record to be the state?)

In [1]:
# A record from a real run
ledger = ExperimentData.from_file('.')
design, result = ledger.to_pandas()

print("evaluations on the record:", len(ledger))
print("stamped by the framework:", [c for c in result.columns if c.startswith('_')])

NameError: name 'ExperimentData' is not defined

## Decisions and what defends them

| decision | defended by |
| :-- | :-- |
| a node is a role, not a pipeline stage | specialization |
| the reviewer runs on a fresh context | safety |
| only one node may close a claim | safety |
| work is handed over in cancellable units | efficiency |
| the state is the data, not the conversation | safety, efficiency |
| one metered path to ground truth | safety, efficiency |
| the model is chosen per node | specialization, efficiency |

## Wiring your own graph

Ours has one node that plans and delegates to the others. That is an accident of our problem,
not a principle: it is a bottleneck and a single point of failure.

Your problem, your graph.

## Admitting a claim

Two mechanisms, neither of them a promise:

* a claim must survive an attempt to refute it
* the deliverable must re-derive its own number from the record

## The falsification charter

One file, injected into every node that judges a claim. §2, in part:

> A search that merely stopped improving [...] is an inadequate test of such a claim [...] failing
> to find a better instance is not the same as showing none exists.

## The reproduction gate

The deliverable is a notebook. Its code cells recompute the result from the record.

It is executed in a clean sandbox before the run may close.

## The study folder

```
my_study/
  PROBLEM_STATEMENT.md   # required: the brief
  config.yaml            # optional: model, budget, how a design is scored
  workspace/
    evaluator.py         # optional: your ground truth
```

## PROBLEM_STATEMENT.md

```markdown
# Minimise a 2-D quadratic

## Objective
Minimise y = (x1 - 1)^2 + (x2 + 2)^2.

## Design space
| variable | type | bounds | units |
|---|---|---|---|
| x1 | continuous | [-5, 5] | dimensionless |
| x2 | continuous | [-5, 5] | dimensionless |
```

Objective, bounds, units. That table is a `Domain`, written in prose.

## config.yaml and the evaluator

```yaml
model: claude-haiku-4-5-20251001
eval_budget: 200
evaluator:
  entrypoint: "workspace/evaluator.py:evaluate"
  output_names: [y]
```

```python
def evaluate(x1: float, x2: float) -> float:
    return (x1 - 1.0) ** 2 + (x2 + 2.0) ** 2
```

## Running a study

```python
from a3dasm import AgenticRun
report = AgenticRun(study_dir="my_study").execute()
```

Back comes `pipeline.ipynb`, the record of every evaluation, and whether the run passed
its gate.

## Lesson 1: Do not overfit the prompt

When a bug is "fixed" by adding a rule to an agent's prompt, that rule must pass a test:

> Would a philosopher of science, reading this rule in isolation, nod at it as a general
> methodological principle, or frown at it as a workaround for one case?

A philosopher nods at: *"A hypothesis cannot be marked SUPPORTED without a falsification attempt
on record."* Popperian; applies to every run that will ever happen.

A philosopher frowns at: *"When calling HypothesisUpdate, pass a single ID, not a comma-separated
list."* That patches one tool-call error and names no principle.

The fallback matters as much as the test. When a failure is real but the obvious rule is overfit,
state the underlying principle instead, or fix it in **code** (a validation, an assertion at the
tool boundary), not in the prompt.

## Lesson 2: "Bulletproof" has a precise meaning

We once let each design space own its own physical data store. Then every place that counts
evaluations had to remember to aggregate across stores.

It didn't: in **seven** places, each found one validation run at a time. Then an eighth: the
design space could be chosen at the call site independently of the delegation, so the
registry-keyed fix was blind whenever the two diverged.

The root cause was not any of the eight sites. The data model did not match the question the whole
codebase asks: *how many evaluations in this run?*

Every aggregation helper was a band-aid: it made the right read available while leaving the wrong
read still present and still looking correct. So the next consumer was born blind.

> A mechanism is bulletproof only when the wrong usage is impossible or loud, not when the right
> usage is merely available.

This generalizes far beyond agents. It is a claim about API design, and you will meet it again the
first time you add an optional argument that callers must remember to pass.

Ask the room: how many of you have written a helper called `get_all_x()` alongside an existing
`get_x()`? That is the shape of the mistake. The fix is not a better helper; it is making
`get_x()` correct so there is nothing to remember.

## Lesson 3: Self-reports are leads, not diagnoses

Every node writes a retrospective when a run closes: what contradicted itself, the most uncertain
decision, what blocked it. These are the highest-signal artifact we have.

And they can be wrong about mechanism. One run blamed a *"silent Abaqus crash"* for a 63%
evaluation failure rate. The raw logs showed the **license server saturating at 16-way
concurrency**.

Same symptom, different fix, and a fix aimed at the reported cause would have achieved nothing
while looking like diligence. Verify the mechanism against the raw logs before acting on it.

In [5]:
# Lesson 3 on the real store: what a report SAYS versus what the record SHOWS.
d = design.join(result)
attempted = d[d.coilable == 1]              # a Riks solve was only ever run on these
failed = ~attempted.riks_converged.astype(bool)

print("solves that never converged:", int(failed.sum()), "of", len(attempted))
print("failure rate:", round(100 * failed.mean(), 1), "%")
print("median wall time, failed vs converged:",
      int(attempted.loc[failed, '_wall_ms'].median() / 1000), "s vs",
      int(attempted.loc[~failed, '_wall_ms'].median() / 1000), "s")

solves that never converged: 154 of 469
failure rate: 32.8 %
median wall time, failed vs converged: 329 s vs 66 s


The retrospective is a lead, not a diagnosis. The record shows the failures running five times
longer than the solves that worked before they die: not a crash, but a wait.

## Lesson 4: Never let a cost argument masquerade as physics

Our problem statement described a solve-time cap as *"a hard property"*, and said a design family
that cannot fit inside it *"is not searchable"*. Both justifications for that cap were **cost**
arguments.

The consequence: a run closed with 5.7 of its 12 hours unspent, reasoning correctly, given what
it had been told, that the one remaining escape required a solver regime the infrastructure
cannot afford.

Presenting a budgeting default as a property of the oracle turned an accounting choice into a
boundary on the design space. The agent then reasoned honestly about a boundary that did not
exist.

The most instructive failure in the project, because nothing malfunctioned. The agent's own
retrospective drew exactly the right distinction: "'My search stopped improving' would not have
justified closing; 'the one remaining mechanism requires a solver regime the infrastructure
cannot afford' is a statement about the space and the tooling, not about my search."

That is better epistemics than most of us apply. It reached a wrong conclusion because *we* wrote
a false premise into the brief. The lesson is about us, not it.

## And one about the brief itself

> Write the problem statement the way a PI would brief a first-year graduate student.

A first-year student has freedom, and asks questions when something is unclear. What the brief
*must* pin down:

* **objective and success criteria**: the headline number or claim
* **design space**: every variable, with bounds, type, and units
* **what "valid" means**: feasibility limits, regimes of validity, thresholds

That middle bullet is a `Domain`. `add_float(name, low, high)`, `add_int`, `add_category`: you
wrote one in Lecture 17, and it is exactly what a good brief has to specify in prose.

What the brief should *not* do is prescribe the method. A brief that specifies the sampler and the
surrogate has hired a technician, not a researcher.

## Summary

* An agentic workflow is a graph of nodes, each a model with tools, choosing its own next step.
* Split the work for **safety**, **specialization** and **efficiency**, and not otherwise.
* The state is the data, not the conversation.
* A claim is admitted by surviving refutation and by reproducing from the record.
* One folder in, one notebook out.

## Next lecture

The same framework, pointed at a real design problem, and what it actually found.

### See you next class

Have fun!